# ml-sgs-turbulence · Quick-Start Notebook

This notebook demonstrates the core workflow:
1. Load a trained baseline MLP
2. Run inference on a MONC NetCDF file
3. Compute skill scores
4. Plot a vertical profile comparison

**Prerequisites**: run the data processor and train at least one model first.
See [docs/training_guide.md](../docs/training_guide.md) for instructions.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from ml_sgs.inference import UnifiedInferenceEngine
from ml_sgs.evaluation import nash_sutcliffe_efficiency, PredictionStorage
from ml_sgs.evaluation.plotting import plot_vertical_profile

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR   = Path("/path/to/netcdf_files")
MODEL_PATH = Path("../models/baseline_mlp/best_model.pt")
SCALER_DIR = Path("../models/baseline_mlp")
NC_FILE    = next(DATA_DIR.glob("*.nc"))  # first file

print(f"Using NetCDF file: {NC_FILE.name}")

In [ ]:
# ── Load engine & run inference ───────────────────────────────────────────
engine = UnifiedInferenceEngine(device="cpu")
engine.add_baseline_models({"MLP": MODEL_PATH}, SCALER_DIR)

predictions, truth = engine.predict_all(NC_FILE, time_idx=0, k_min=2, k_max=40)

visc_pred = predictions["Baseline_MLP"]["visc_coeff"]
visc_true = truth["visc_coeff"]

print(f"NSE (viscosity): {nash_sutcliffe_efficiency(visc_pred, visc_true):.4f}")

In [ ]:
# ── Vertical profile plot ─────────────────────────────────────────────────
fig = plot_vertical_profile(
    predictions={"Baseline MLP": visc_pred},
    truth=visc_true,
    variable="visc_coeff",
)
plt.show()